# rabi prefix raw data
See the repository README for the paper configurations and cached-data reproduction commands.


In [ ]:
from __future__ import annotations

from dataclasses import fields, replace
from pathlib import Path
import hashlib
import json
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import chi2, ncx2, norm, poisson


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "dm_search").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing src/dm_search")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from dm_search.rabi import (  # noqa: E402
    RabiExperimentConfig,
    RabiParticleSpec,
    RabiReadoutConfig,
    build_rabi_particle_grid,
    longitudinal_relaxation_factor,
    rabi_oracle_known_signal_power,
    sample_rabi_diagnostics_at_checkpoints,
    transverse_relaxation_factor,
    with_true_phase_amplitude,
)
from dm_search.statistics import null_threshold  # noqa: E402
from dm_search.rabi_prefix_cache import (  # noqa: E402
    build_rabi_prefix_setup,
    rabi_prefix_cache_path,
)


plt.style.use("seaborn-v0_8-whitegrid")

# Shared manuscript typography and frameless legends.
plt.style.use(REPO_ROOT / "plot_final" / "rabi_paper.mplstyle")


In [ ]:
# Set False in both prefix_raw_data and discovery_probability for production.
SMALL_CHECK = False
RUN_PRESET = "final" if SMALL_CHECK else os.environ.get("RABI_PREFIX_PRESET", "final")
DETECTOR_PROFILE = os.environ.get("RABI_PREFIX_DETECTOR_PROFILE", "current")
os.environ["RABI_PREFIX_DETECTOR_PROFILE"] = DETECTOR_PROFILE
os.environ.setdefault("RABI_PREFIX_ADAPTIVE_PERIODS", "1,2")
os.environ.setdefault("RABI_PREFIX_INCLUDE_STEIN", "0")
PREFIX_ENVIRON = dict(os.environ)
if SMALL_CHECK:
    DETECTOR_PROFILE = "current"
    PREFIX_ENVIRON.update({
        "RABI_PREFIX_DETECTOR_PROFILE": DETECTOR_PROFILE,
        "RABI_PREFIX_N_EXPERIMENTS": "128",
        "RABI_PREFIX_N_NULL": "256",
        "RABI_PREFIX_BATCH_SIZE": "64",
        "RABI_PREFIX_AMPLITUDE_COUNT": "12",
        "RABI_PREFIX_DRIVE_PHASE_COUNT": "24",
        "RABI_PREFIX_CANDIDATE_ANGLE_COUNT": "8",
        "RABI_PREFIX_CANDIDATE_POLAR_COUNT": "5",
        "RABI_PREFIX_ADAPTIVE_PERIODS": "1",
    })
PREFIX_SETUP = build_rabi_prefix_setup(RUN_PRESET, environ=PREFIX_ENVIRON)

ALPHA = PREFIX_SETUP.alpha
TARGET_POWERS = PREFIX_SETUP.target_powers
N_SHOT_MAX = PREFIX_SETUP.n_shot_max
N_NULL = PREFIX_SETUP.n_null
N_SIGNAL = PREFIX_SETUP.n_signal
BATCH_SIZE = PREFIX_SETUP.batch_size
PHASE_GRID = PREFIX_SETUP.phase_grid
CHECKPOINTS = PREFIX_SETUP.checkpoints
PHASE_AVERAGE_MODE = PREFIX_SETUP.phase_average_mode
STRATEGIES = PREFIX_SETUP.strategies
base_template = PREFIX_SETUP.base_template
particle_grid = PREFIX_SETUP.particle_grid
TRUE_DRIVE_PHASES = particle_grid.drive_phase_values

OUTPUT_BASENAME = "rabi_prefix_sensitivity_vs_shots"
CACHE_DIR = REPO_ROOT / "plot_final" / "cache"
FIGURE_DIR = REPO_ROOT / "plot_final" / "figure"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
USE_CACHE = os.environ.get("RABI_PREFIX_USE_CACHE", "1") != "0"
FORCE_RECOMPUTE = os.environ.get("RABI_PREFIX_FORCE_RECOMPUTE", "0") == "1"
RAW_CACHE_OVERRIDE = os.environ.get("RABI_PREFIX_RAW_CACHE", "").strip()

raw_setup = PREFIX_SETUP.raw_setup
analysis_setup = PREFIX_SETUP.analysis_setup
setup = {**raw_setup, **analysis_setup}
print(json.dumps(setup, indent=2))
print("raw cache hash excludes alpha and target_powers; changing them reuses the same simulation cache")


In [ ]:
def cache_path() -> Path:
    return rabi_prefix_cache_path(CACHE_DIR, RUN_PRESET, raw_setup, basename=OUTPUT_BASENAME)


def h1_phase_averaged_checkpoint_diagnostics(
    config: RabiExperimentConfig,
    strategy: str,
    phase_amplitude: float,
    total_count: int,
    seed_base: int,
) -> tuple[dict[str, np.ndarray], np.ndarray]:
    sampler_strategy = STRATEGIES[strategy].get("sampler_strategy", strategy)
    if strategy == "fixed_z":
        fixed_config = replace(with_true_phase_amplitude(config, phase_amplitude), true_drive_phase=0.0)
        diagnostics = sample_rabi_diagnostics_at_checkpoints(
            fixed_config,
            sampler_strategy,
            "H1",
            total_count,
            CHECKPOINTS,
            batch_size=BATCH_SIZE,
            seed=seed_base,
        )
        return diagnostics, np.zeros(total_count, dtype=float)

    if PHASE_AVERAGE_MODE == "random_per_experiment":
        phase_rng = np.random.default_rng(seed_base + 91_337)
        random_true_drive_phases = phase_rng.uniform(0.0, 2.0 * np.pi, total_count)
        adaptive_config = replace(with_true_phase_amplitude(config, phase_amplitude), true_drive_phase=0.0)
        diagnostics = sample_rabi_diagnostics_at_checkpoints(
            adaptive_config,
            sampler_strategy,
            "H1",
            total_count,
            CHECKPOINTS,
            batch_size=BATCH_SIZE,
            seed=seed_base,
            true_drive_phases=random_true_drive_phases,
        )
        return diagnostics, random_true_drive_phases

    per_phase = int(np.ceil(total_count / len(TRUE_DRIVE_PHASES)))
    diagnostic_pieces = {
        "log_bayes_factors": [],
        "predictable_kl": [],
        "predictable_variance": [],
        "polarization_design": [],
    }
    phase_pieces = []
    for phase_index, drive_phase in enumerate(TRUE_DRIVE_PHASES):
        adaptive_config = replace(
            with_true_phase_amplitude(config, phase_amplitude),
            true_drive_phase=float(drive_phase),
        )
        piece = sample_rabi_diagnostics_at_checkpoints(
            adaptive_config,
            sampler_strategy,
            "H1",
            per_phase,
            CHECKPOINTS,
            batch_size=min(BATCH_SIZE, per_phase),
            seed=seed_base + phase_index,
        )
        for key in diagnostic_pieces:
            diagnostic_pieces[key].append(piece[key])
        phase_pieces.append(np.full(per_phase, float(drive_phase)))
    diagnostics = {
        key: np.concatenate(pieces, axis=0)[:total_count]
        for key, pieces in diagnostic_pieces.items()
    }
    return diagnostics, np.concatenate(phase_pieces)[:total_count]


def run_raw_checkpoint_scan() -> dict:
    start = time.perf_counter()
    config = replace(base_template, n_shots=N_SHOT_MAX)
    raw = {
        "raw_setup": raw_setup,
        "checkpoints": CHECKPOINTS.copy(),
        "phase_grid": PHASE_GRID.copy(),
        "strategies": list(STRATEGIES),
        "logB_h0": {},
        "logB_h1": {},
        "true_drive_phases_h1": {},
        "predictable_kl_h0": {},
        "predictable_variance_h0": {},
        "polarization_design_h0": {},
        "polarization_design_h1": {},
    }

    for strategy_index, strategy in enumerate(STRATEGIES):
        # Use common random numbers for the two adaptive policies.
        seed_index = 3 if strategy.startswith("adaptive") else strategy_index
        sampler_strategy = STRATEGIES[strategy].get("sampler_strategy", strategy)
        null_diagnostics = sample_rabi_diagnostics_at_checkpoints(
            config,
            sampler_strategy,
            "H0",
            N_NULL,
            CHECKPOINTS,
            batch_size=BATCH_SIZE,
            seed=10_000 + 100 * seed_index,
        )
        raw["logB_h0"][strategy] = np.asarray(
            null_diagnostics["log_bayes_factors"], dtype=np.float64
        )
        raw["predictable_kl_h0"][strategy] = np.asarray(
            null_diagnostics["predictable_kl"], dtype=np.float64
        )
        raw["predictable_variance_h0"][strategy] = np.asarray(
            null_diagnostics["predictable_variance"], dtype=np.float64
        )
        raw["polarization_design_h0"][strategy] = np.asarray(
            null_diagnostics["polarization_design"], dtype=np.float64
        )

        h1_samples = np.empty((PHASE_GRID.size, N_SIGNAL, CHECKPOINTS.size), dtype=np.float64)
        h1_true_phases = np.empty((PHASE_GRID.size, N_SIGNAL), dtype=np.float64)
        h1_design = np.empty(
            (PHASE_GRID.size, N_SIGNAL, CHECKPOINTS.size, 3),
            dtype=np.float64,
        )
        for phase_index, phase_amplitude in enumerate(PHASE_GRID):
            print(f"{strategy}: H1 Phi_R={phase_amplitude:g} ({phase_index + 1}/{PHASE_GRID.size})", flush=True)
            diagnostics, true_phases = h1_phase_averaged_checkpoint_diagnostics(
                config,
                strategy,
                float(phase_amplitude),
                N_SIGNAL,
                seed_base=20_000 + 10_000 * seed_index + 100 * phase_index,
            )
            h1_samples[phase_index] = np.asarray(
                diagnostics["log_bayes_factors"], dtype=np.float64
            )
            h1_design[phase_index] = np.asarray(
                diagnostics["polarization_design"], dtype=np.float64
            )
            h1_true_phases[phase_index] = np.asarray(true_phases, dtype=np.float64)
        raw["logB_h1"][strategy] = h1_samples
        raw["true_drive_phases_h1"][strategy] = h1_true_phases
        raw["polarization_design_h1"][strategy] = h1_design
        print(f"{strategy}: stored logB and trajectory diagnostics", flush=True)

    elapsed = time.perf_counter() - start
    raw["elapsed_seconds"] = float(elapsed)
    print(f"elapsed_seconds={elapsed:.1f}")
    return raw


def save_raw_scan(path: Path, raw: dict) -> None:
    arrays = {
        "raw_setup": json.dumps(raw["raw_setup"]),
        "checkpoints": np.asarray(raw["checkpoints"], dtype=np.int64),
        "phase_grid": np.asarray(raw["phase_grid"], dtype=np.float64),
        "strategies": np.asarray(raw["strategies"], dtype=str),
        "elapsed_seconds": np.asarray(raw.get("elapsed_seconds", np.nan), dtype=np.float64),
    }
    for strategy in raw["strategies"]:
        arrays[f"logB_h0__{strategy}"] = np.asarray(raw["logB_h0"][strategy], dtype=np.float64)
        arrays[f"logB_h1__{strategy}"] = np.asarray(raw["logB_h1"][strategy], dtype=np.float64)
        arrays[f"true_drive_phases_h1__{strategy}"] = np.asarray(
            raw["true_drive_phases_h1"][strategy], dtype=np.float64
        )
        arrays[f"predictable_kl_h0__{strategy}"] = np.asarray(
            raw["predictable_kl_h0"][strategy], dtype=np.float64
        )
        arrays[f"predictable_variance_h0__{strategy}"] = np.asarray(
            raw["predictable_variance_h0"][strategy], dtype=np.float64
        )
        arrays[f"polarization_design_h0__{strategy}"] = np.asarray(
            raw["polarization_design_h0"][strategy], dtype=np.float64
        )
        arrays[f"polarization_design_h1__{strategy}"] = np.asarray(
            raw["polarization_design_h1"][strategy], dtype=np.float64
        )
    np.savez_compressed(path, **arrays)


def load_raw_scan(path: Path) -> dict:
    payload = np.load(path, allow_pickle=True)
    strategies = [str(item) for item in payload["strategies"].tolist()]
    required_prefixes = (
        "predictable_kl_h0",
        "predictable_variance_h0",
        "polarization_design_h0",
        "polarization_design_h1",
    )
    missing = [
        f"{prefix}__{strategy}"
        for strategy in strategies
        for prefix in required_prefixes
        if f"{prefix}__{strategy}" not in payload.files
    ]
    if missing:
        raise ValueError(
            "The selected cache predates the trajectory-diagnostic schema. "
            "Regenerate it with this notebook. Missing keys: "
            + ", ".join(missing[:6])
        )
    raw = {
        "raw_setup": json.loads(str(payload["raw_setup"])),
        "checkpoints": np.asarray(payload["checkpoints"], dtype=np.int64),
        "phase_grid": np.asarray(payload["phase_grid"], dtype=np.float64),
        "strategies": strategies,
        "logB_h0": {},
        "logB_h1": {},
        "true_drive_phases_h1": {},
        "predictable_kl_h0": {},
        "predictable_variance_h0": {},
        "polarization_design_h0": {},
        "polarization_design_h1": {},
        "elapsed_seconds": float(payload["elapsed_seconds"])
        if "elapsed_seconds" in payload.files
        else float("nan"),
    }
    for strategy in strategies:
        raw["logB_h0"][strategy] = np.asarray(payload[f"logB_h0__{strategy}"], dtype=np.float64)
        raw["logB_h1"][strategy] = np.asarray(payload[f"logB_h1__{strategy}"], dtype=np.float64)
        raw["true_drive_phases_h1"][strategy] = np.asarray(
            payload[f"true_drive_phases_h1__{strategy}"], dtype=np.float64
        )
        raw["predictable_kl_h0"][strategy] = np.asarray(
            payload[f"predictable_kl_h0__{strategy}"], dtype=np.float64
        )
        raw["predictable_variance_h0"][strategy] = np.asarray(
            payload[f"predictable_variance_h0__{strategy}"], dtype=np.float64
        )
        raw["polarization_design_h0"][strategy] = np.asarray(
            payload[f"polarization_design_h0__{strategy}"], dtype=np.float64
        )
        raw["polarization_design_h1"][strategy] = np.asarray(
            payload[f"polarization_design_h1__{strategy}"], dtype=np.float64
        )
    return raw


In [ ]:
if RAW_CACHE_OVERRIDE:
    path = Path(RAW_CACHE_OVERRIDE).expanduser().resolve()
else:
    path = cache_path()

if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
    print(f"raw logB/diagnostic cache already exists: {path}")
    raw_scan = load_raw_scan(path)
else:
    raw_scan = run_raw_checkpoint_scan()
    if USE_CACHE:
        save_raw_scan(path, raw_scan)
        print(f"saved raw logB/diagnostic cache {path}")

summary = {
    "raw_cache_path": str(path),
    "raw_setup": raw_scan["raw_setup"],
    "elapsed_seconds": float(raw_scan.get("elapsed_seconds", float("nan"))),
    "checkpoints": np.asarray(raw_scan["checkpoints"], dtype=int).tolist(),
    "phase_grid": np.asarray(raw_scan["phase_grid"], dtype=float).tolist(),
    "shapes": {
        strategy: {
            "h0": list(np.asarray(raw_scan["logB_h0"][strategy]).shape),
            "h1": list(np.asarray(raw_scan["logB_h1"][strategy]).shape),
            "predictable_kl_h0": list(
                np.asarray(raw_scan["predictable_kl_h0"][strategy]).shape
            ),
            "predictable_variance_h0": list(
                np.asarray(raw_scan["predictable_variance_h0"][strategy]).shape
            ),
            "polarization_design_h0": list(
                np.asarray(raw_scan["polarization_design_h0"][strategy]).shape
            ),
            "polarization_design_h1": list(
                np.asarray(raw_scan["polarization_design_h1"][strategy]).shape
            ),
        }
        for strategy in raw_scan["strategies"]
    },
}
print(json.dumps(summary, indent=2))
